In [0]:
import sys
import os
import gc
sys.path.append(os.path.join(os.path.dirname(os.getcwd()), 'src'))

import yaml
import pandas as pd
import numpy as np

from data_pull.loaders import (
    load_user_table,
    load_task_complete_table,
    load_respondent_info_table,
    load_task_table,
    load_ditr_table,
    load_all_wonky_studies,
    load_stripe_verification
)
from data_pull.joiners import (  
    join_user_task_respondent,
    join_wonky_balance_with_task,
    merge_wonky_data_spark,  
    prepare_final_output,
    deduplicate_spark_columns,
    join_stripe_verification
)
from data_pull.aggregators import (  
    enrich_user_info_with_task_counts,
    union_wonky_study_dataframes,
    create_wonky_respondent_df_spark,
    create_wonky_respondent_summary_spark,
    calculate_wonky_task_ratio_spark
)

with open('../configs/data_paths.yaml', 'r') as f:
    paths_config = yaml.safe_load(f)

with open('../configs/wonky_studies.yaml', 'r') as f:
    wonky_config = yaml.safe_load(f)

try:
    spark.catalog.clearCache()
except Exception:
    pass

print("Setup complete")

Specify Paths and Load Data from Core Tables

In [0]:
SILVER_PATH = paths_config['silver_path']
COUNTRY = paths_config['filters']['country']
MIN_DATE = paths_config['filters']['min_date']
TASK_ORIGIN = paths_config['filters']['task_origin']
PROJECT_REPO_PATH = paths_config['project_repository_path']

print(f"Country: {COUNTRY}")
print(f"Min date: {MIN_DATE}")
print(f"Task origin: {TASK_ORIGIN}")

In [0]:
s_user = load_user_table(spark, SILVER_PATH, country=COUNTRY)
s_task_complete = load_task_complete_table(
    spark, SILVER_PATH, min_date=MIN_DATE, task_origin=TASK_ORIGIN
)
s_respondent_info = load_respondent_info_table(
    spark,
    SILVER_PATH,
    country=COUNTRY,
    exclude_cols=wonky_config.get("cols_to_exclude_respondent_info", []),
)
s_ditr = load_ditr_table(spark, SILVER_PATH, user_df=s_user)
s_task = load_task_table(spark, SILVER_PATH)
stripe_df = load_stripe_verification(spark, SILVER_PATH)

print(f"Users: {s_user.count():,}")
print(f"Task completions: {s_task_complete.count():,}")
print(f"Respondent info: {s_respondent_info.count():,}")
print(f"DITR records: {s_ditr.count():,}")
print(f"Tasks: {s_task.count():,}")
print(f"Stripe: {stripe_df.count():,}")

Join Core Tables

In [0]:
user_info_spark = join_user_task_respondent(
    s_user, s_task_complete, s_respondent_info, ditr_df=s_ditr
)

user_info_spark = enrich_user_info_with_task_counts(user_info_spark)

user_info_spark = user_info_spark.join(
    s_task.select("task_pk", "task_length_of_task"),
    user_info_spark.taskPk == s_task.task_pk,
    "left",
)

user_info_spark = join_stripe_verification(user_info_spark, stripe_df)

In [0]:
user_info_spark = deduplicate_spark_columns(user_info_spark)

user_info_spark.cache()

print(f"Joined records: {user_info_spark.count():,}")
print(
    f"Unique respondents: {user_info_spark.select('respondentPk').distinct().count():,}"
)
print(f"Columns: {len(user_info_spark.columns)}")

Load and Join Balance Tables (Wonky Study Set)

In [0]:
def process_wonky_studies_spark(
    spark, uuids, task_df, base_path, cols_to_include, wonky_flag
):
    from pyspark.sql.functions import lit

    balance_dfs, failed = load_all_wonky_studies(
        spark,
        uuids,
        base_path=base_path,
        cols_to_include=cols_to_include,
        verbose=True,
        max_workers=4,
    )

    if not balance_dfs:
        return None

    joined_dfs = [join_wonky_balance_with_task(df, task_df) for df in balance_dfs]

    wonky_spark = union_wonky_study_dataframes(joined_dfs)

    wonky_spark = wonky_spark.withColumn("wonky_study_count", lit(wonky_flag))

    print(
        f"Loaded {len(balance_dfs)}/{len(uuids)} studies | {wonky_spark.select('respondent_pk').distinct().count():,} respondents"
    )

    return wonky_spark

In [0]:
uuids_wonky = [
    u for u, is_wonky in wonky_config["wonky_study_uuids_updated"].items() if is_wonky
]
uuids_control = [
    u
    for u, is_wonky in wonky_config["wonky_study_uuids_updated"].items()
    if not is_wonky
]

print("Processing wonky studies...")
wonky_map_exposed = process_wonky_studies_spark(
    spark,
    uuids_wonky,
    s_task,
    PROJECT_REPO_PATH,
    wonky_config["cols_to_include_subset"],
    wonky_flag=1,
)

print("Processing control studies...")
wonky_map_control = process_wonky_studies_spark(
    spark,
    uuids_control,
    s_task,
    PROJECT_REPO_PATH,
    wonky_config["cols_to_include_subset"],
    wonky_flag=0,
)

In [0]:
wonky_map_spark = wonky_map_exposed.unionByName(wonky_map_control, allowMissingColumns=True)
wonky_map_spark.cache()

print(f"\nTotal wonky map: {wonky_map_spark.count():,} records | {wonky_map_spark.select('respondent_pk').distinct().count():,} respondents")

In [0]:
wonky_respondent_spark = create_wonky_respondent_df_spark(
    wonky_map_spark,
    group_cols=wonky_config['cols_to_group'],
    cols_to_drop=wonky_config.get('cols_to_drop', [])
)

wonky_respondent_spark.cache()
print(f"Wonky respondent-level: {wonky_respondent_spark.count():,} records")

Merge and Save

In [0]:
user_info_final_spark = merge_wonky_data_spark(
    user_info_spark,
    wonky_respondent_spark,
    user_respondent_col="respondentPk",
    user_task_col="taskPk",
    wonky_respondent_col="balance_respondentPk",
    wonky_task_col="task_pk"
)

user_info_final_spark.cache()
print(f"Final Spark DataFrame: {user_info_final_spark.count():,} rows")

In [0]:
from pyspark.sql.functions import col

balance_users_spark = user_info_final_spark.filter(col("survey_type").isNotNull())

In [0]:
# running a sampling to reduce overall size for speed
rest_of_users_spark = (
    user_info_final_spark
    .filter(col("survey_type").isNull())
    .sample(fraction=0.3, seed=42)  
    .limit(250000)                   
)

In [0]:
user_info_sampled_spark = balance_users_spark.unionByName(
    rest_of_users_spark, 
    allowMissingColumns=True
)

In [0]:
sorted(user_info_sampled_spark.columns)

In [0]:
wonky_summary_spark = create_wonky_respondent_summary_spark(
    wonky_respondent_spark,
    respondent_id_col="balance_respondentPk",
    categorical_cols=wonky_config.get('summary_categorical_cols', None)
)

wonky_counts_spark = calculate_wonky_task_ratio_spark(
    user_info_spark,
    wonky_summary_spark,
    user_respondent_col="respondentPk",
    wonky_respondent_col="balance_respondentPk"
)

print(f"Summary records: {wonky_summary_spark.count():,}")
print(f"Wonky counts records: {wonky_counts_spark.count():,}")

In [0]:
ESSENTIAL_COLUMNS = [
    i for i in user_info_final_spark.columns if i not in wonky_config["cols_to_drop"]
]

In [0]:
print("Converting to pandas with column selection...")
user_info_final = prepare_final_output(
    user_info_sampled_spark,
    columns_to_keep=ESSENTIAL_COLUMNS
)

In [0]:
wonky_respondent_df = wonky_respondent_spark.toPandas()
wonky_summary = wonky_summary_spark.toPandas()
wonky_counts = wonky_counts_spark.toPandas()
wonky_map = wonky_map_spark.toPandas()

print(f"\nFinal dataset shape: {user_info_final.shape}")
print(f"Memory usage: {user_info_final.memory_usage(deep=True).sum() / 1e9:.2f} GB")

In [0]:
from data_pull.joiners import validate_merge_equivalence

validation = validate_merge_equivalence(user_info_final)
print(f"Validation passed: {validation['passed']}")
if validation['missing_cols']:
    print(f"Missing columns: {validation['missing_cols']}")
print(f"Checks: {validation['checks']}")

print(f"\n--- Sanity Checks ---")
print(f"Unique respondents: {user_info_final['respondentPk'].nunique():,}")
print(f"Records with wonky exposure: {(user_info_final['wonky_study_count'] > 0).sum():,}")
print(f"wonky_user_flag distribution: {user_info_final['wonky_user_flag'].value_counts().to_dict()}")

if '_merge' in user_info_final.columns:
    print(f"_merge distribution: {user_info_final['_merge'].value_counts().to_dict()}")

assert user_info_final['wonky_study_count'].isna().sum() == 0, "ERROR: Null values in wonky_study_count!"
assert user_info_final['wonky_user_flag'].isna().sum() == 0, "ERROR: Null values in wonky_user_flag!"
print("All validations passed!")


In [0]:
user_info_spark.unpersist()
user_info_final_spark.unpersist()
wonky_map_spark.unpersist()
wonky_respondent_spark.unpersist()

In [0]:
spark.catalog.clearCache()

gc.collect()

In [0]:
notebook_path = os.getcwd()
repo_root = os.path.abspath(os.path.join(notebook_path, ".."))
misc_dir = os.path.join(repo_root, "misc")
os.makedirs(misc_dir, exist_ok=True)

output_path = os.path.join(
    misc_dir, os.path.basename(paths_config["output_files"]["user_info_df"])
)

In [0]:
sorted(user_info_final.columns)

In [0]:
user_info_final.to_parquet(output_path, index=False)

print("Files saved successfully:")
print(f"  - {output_path}")